<a href="https://colab.research.google.com/github/ancestor9/Listeria/blob/main/Listeria_03_Extract_Data_predict.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
path ='/content/drive/MyDrive/MFDS/'
df = pd.read_parquet(path + "listeria_dataset.parquet")
print(len(df))

676531


In [ ]:
def summarize_df(df: pd.DataFrame) -> pd.DataFrame:
    # 데이터프레임의 각 컬럼별 데이터 타입, 결측치 수/비율, 대표값 5개를 요약한 DataFrame을 반환합니다.
    summary_data = []
    for col in df.columns:
        dtype = df[col].dtype
        n_unique = df[col].nunique()
        null_cnt = df[col].isnull().sum()
        null_ratio = round((null_cnt / len(df)) * 100, 2)
        unique_vals = df[col].dropna().unique()
        sample_vals = list(unique_vals[:7])

        summary_data.append({
            'Column': col,
            'N_Unique': n_unique,
            'Dtype': dtype,
            'Null_Count': null_cnt,
            'Null_Ratio(%)': null_ratio,
            'Sample_Values_5': sample_vals
        })

    return pd.DataFrame(summary_data)

In [ ]:
summarize_df(df).to_excel('/content/listeria_dataset_summary.xlsx', index=False)

In [ ]:
selection_list = pd.read_excel(path + "listeria_dataset_summary_selection.xlsx")
selection_list

,Column,feature 포함여부,N_Unique,Dtype,Null_Count,Null_Ratio(%),Sample_Values_5
0,lims_data_key,0,676531,string,0,0.00,"['O002012_250100044_1_5', 'O000728_250103704_4..."
1,hazard_label,0,1,category,0,0.00,['리스테리아']
2,judge_result,1,2,category,0,0.00,"['불검출', '검출']"
3,src_route,1,2,category,0,0.00,"['domestic', 'overseas']"
4,data_source,0,1,category,0,0.00,['식약처']
5,expr_iem_nm,0,4,category,0,0.00,"['리스테리아모노사이토제네스', '리스테리아 모노사이토제네스', '리스테리아 모노싸..."
6,result_value,0,100,string,0,0.00,"['음성,음성,음성,음성,음성', '음성', '불검출', '음성, 음성, 음성, 음..."
7,result_value_num,0,2,float32,672754,99.44,"[np.float32(0.0), np.float32(2.0)]"
8,judge_stdr,0,17,string,669258,98.92,"['n=5,c=0,m=0/25g', 'n=5, c=0, m=0/25g', '음성 ..."
9,judge_stdr_num,0,5,float32,676463,99.99,"[np.float32(100.0), np.float32(20.0), np.float..."


In [ ]:
selection_list[selection_list['feature 포함여부'] == 1]['Column'].unique()

array(['judge_result', 'src_route', 'inspct_instt_code',
       'train_prdlst_nm', 'train_prdlst_nm_2', 'prduct_nm', 'mnfctur_de',
       'addr', 'lat', 'lon'], dtype=object)

In [ ]:
df = df[selection_list[selection_list['feature 포함여부'] == 1]['Column'].unique()].copy()
df

,judge_result,src_route,inspct_instt_code,train_prdlst_nm,train_prdlst_nm_2,prduct_nm,mnfctur_de,addr,lat,lon
0,불검출,domestic,O002012,즉석식품류,즉석섭취.편의식품류,렌위치 NY 샌드위치,2025-10-11,경기도 안성시 미양면 제2공단2길 25,36.975372,127.215195
1,불검출,domestic,O000728,기타기준규격외,기타기준규격외,"삼치(국내산_해동,전처리,세척전)",2025-04-28,전라남도 여수시 화양면 석교로 121,34.708126,127.613495
2,불검출,domestic,O000130,유가공품류,산양유,숲속산양목장 산양유,2025-06-09,전북특별자치도 임실군 신덕면 수지로 73-8,35.683971,127.178284
3,불검출,overseas,1471124,유가공품류,치즈류,무게또 고르곤졸라 & 마스카포네,2025-07-25,"VIA V.EMANUELE 62, 23815 INTROBIO (LECCO)",45.977798,9.449053
4,불검출,domestic,O000168,빙과류,아이스크림류,뉴 쿠즈코초콜릿,2025-08-11,경기도 광주시 오포읍 봉골길33번길 23,37.362312,127.213516
...,...,...,...,...,...,...,...,...,...,...
676526,불검출,domestic,O000858,기타기준규격외,기타기준규격외,별별생선까스 동결 후,2025-05-19,경기도 광주시 초월읍 숯골길 17-42(1층),37.385250,127.287964
676527,불검출,domestic,O000858,알가공품류,알가공품,에코팜 금 계란말이,2025-06-25,경기도 김포시 대곶면 대곶로 429-19,37.649372,126.582138
676528,불검출,domestic,O000169,기타기준규격외,기타기준규격외,편육 냉각,2025-06-16,경기도 파주시 조리읍 등원로330번길 110,37.744717,126.805168
676529,불검출,domestic,O000728,기타기준규격외,기타기준규격외,유기농레몬즙100%(87℃/55분살균후),2025-07-17,충청남도 금산군 군북면 군북로 586,36.168907,127.527725


### 의문사항
- result_value의 "음성,음성,음성,음성,음성" 의미는?
- expr_iem_nm, legal_dong_code(47641개 결측치는 모두 해외임), legal_dong_code(3772개)는 시군단위 하위 지역정보 로 제거

In [ ]:
df['judge_result'].value_counts(normalize=True)*100

,proportion
judge_result,
불검출,99.708956
검출,0.291044


In [ ]:

df['src_route'].value_counts()

,count
src_route,
domestic,628889
overseas,47642


## 검출된 데이터만 가지고 학습(가정)
- 즉 과거의 불검출된 품목은 모두 위험이 없다고 가정함
> - **품목기준 : train_prdlst_code_2    하위 품목군 코드 = 품목군_3단계**

In [ ]:
# @title 식품종류별로 검사결과 검출이 된 품목과 검측률
df['judge_result'] = df['judge_result'].map({'검출': 1, '불검출': 0}).astype(int)
product_list_tm = df.groupby('train_prdlst_nm_2')['judge_result'].mean().sort_values(ascending=False)*100
product_list_tm = product_list_tm[product_list_tm != 0]
product_list_tm

/tmp/ipykernel_1213/2712643283.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  product_list_tm = df.groupby('train_prdlst_nm_2')['judge_result'].mean().sort_values(ascending=False)*100


,judge_result
train_prdlst_nm_2,
팽이버섯,25.000000
엄나무,6.250000
포장육,3.785607
기타,2.223303
포유류,1.990812
식육함유가공품,0.952381
해양어류,0.950508
기타 수산물가공품,0.859873
갑각류,0.699301


In [ ]:
print(f'검출율이 "0"이 아닌 식품종류 개수는 {len(product_list_tm)} 종류')

검출율이 "0"이 아닌 식품종류 개수는 32 종류


In [ ]:
# @title 리스테리아가 검출된 품목만 선택하여 데이터 구성
product_list_non_tm = product_list_tm[product_list_tm == 0]
print(product_list_non_tm.index)
tf = df[df['train_prdlst_nm_2'].isin(product_list_tm.index)].copy()
print("tf['train_prdlst_nm_2'] 열(Column)에 들어있는 상품명 중 product_list_non_tm.index에 존재하는 상품명이 단 하나라도 있는가?")
tf['train_prdlst_nm_2'].isin(product_list_non_tm.index).any()

CategoricalIndex([], categories=['NAG(엔에이지, N-아세틸글루코사민, N-Acetylglucosamine)', '가공김(조미김 또는 구운김)', '가공두부', '가공유류', ..., '효모식품', '효모추출물', '효소식품', '효소처리스테비아'], ordered=False, dtype='category', name='train_prdlst_nm_2')
tf['train_prdlst_nm_2'] 열(Column)에 들어있는 상품명 중 product_list_non_tm.index에 존재하는 상품명이 단 하나라도 있는가?


np.False_

In [ ]:
# @title 식품종류별로 수입 및 국내를 구분하여 검출룰을 판단
detection_rate_by_product_route = (
    tf.groupby(['train_prdlst_nm_2', 'src_route'], observed=True)['judge_result']
    .mean()
    .unstack('src_route')
    .fillna(0) * 100
)

# domestic(국내) 수치 기준으로 내림차순 정렬하여 출력
detection_rate_by_product_route.sort_values(by='domestic', ascending=False)

src_route,domestic,overseas
train_prdlst_nm_2,,
팽이버섯,25.000000,0.000000
엄나무,6.250000,0.000000
포장육,3.785607,0.000000
기타,2.223612,0.000000
포유류,1.143583,3.275530
식육함유가공품,0.952381,0.000000
기타 수산물가공품,0.920873,0.000000
해양어류,0.862069,0.954003
갑각류,0.800000,0.000000


# 2026_10_7

1. 예측 기간 변경

- 기존: 11일 예측

- 변경: 60일 예측



2. 예측 결과 단위 변경

- 기존: 검체 단위 예측

- 변경: 지역 × 품목군_2단계 × 품목군_3단계 단위로 결과 산출

  (단, 가공식품의 경우 품목군_3단계 × 품목군_4단계)



같은 지역·같은 품목군이라도 하위 품목군이 다르면 각각 별도의 예측 단위로 구분됩니다.



예를 들어 "경기도 안성시 × 견과종실류"만으로는 하나의 단위가 아니고,

"경기도 안성시 × 견과종실류(2단계) × 땅콩 또는 견과류(3단계)"와

"경기도 안성시 × 견과종실류(2단계) × 유지종실류(3단계)"가 각각 별개의 예측 단위가 되어 확률도 따로 산출됩니다.


■ 추가 컬럼

> train_prdlst_code_2    하위 품목군 코드 = 품목군_3단계

>  train_prdlst_nm_2      하위 품목군명



- 기존 품목군 = 2단계(train_prdlst_code)보다 한 단계 아래의 분류입니다.

  하위 단계가 없는 품목은 기존 품목군 값을 그대로 넣었습니다.

>  (단, 가공식품의 경우 품목군_3단계 × 품목군_4단계) 가공식품은 코드가 C로 시작하는 품목입니다.

## **데이터 전처리**

### **1. addr(주소) 데이터 전처리**

- 지역정보 feature를 위해 해외데이터는 제거

In [ ]:
tf['src_route'].value_counts()

,count
src_route,
domestic,499303
overseas,35277


In [ ]:
tf = tf[tf['src_route'] == 'domestic'].copy()
print(len(tf))

499303


In [ ]:
# 'addr' 컬럼의 문자열을 공백 기준으로 분할하여 '시도'와 '시군구'를 추출합니다.
# 데이터에 결측치나 비정상적인 주소가 있을 수 있으므로 string 타입으로 변환 후 처리합니다.
tf['sido'] = tf['addr'].astype(str).apply(lambda x: x.split()[0] if len(x.split()) > 0 else '')
tf['sigungu'] = tf['addr'].astype(str).apply(lambda x: x.split()[1] if len(x.split()) > 1 else '')
del tf['addr']
del tf['src_route']
tf

,judge_result,inspct_instt_code,train_prdlst_nm,train_prdlst_nm_2,prduct_nm,mnfctur_de,lat,lon,sido,sigungu
0,0,O002012,즉석식품류,즉석섭취.편의식품류,렌위치 NY 샌드위치,2025-10-11,36.975372,127.215195,경기도,안성시
1,0,O000728,기타기준규격외,기타기준규격외,"삼치(국내산_해동,전처리,세척전)",2025-04-28,34.708126,127.613495,전라남도,여수시
5,0,O000473,기타기준규격외,기타기준규격외,돈육(냉동)(국내)절단/성형후1차,2025-10-15,37.823318,127.138603,경기도,포천시
7,0,O000392,기타기준규격외,기타기준규격외,표고버섯분말 살균후 (완제품),2025-05-23,36.444393,126.832138,충청남도,청양군
9,0,O000858,기타기준규격외,기타기준규격외,갑오징어-원재료,2025-08-25,35.991085,126.689377,전북특별자치도,군산시
...,...,...,...,...,...,...,...,...,...,...
676524,0,O000728,기타기준규격외,기타기준규격외,쌀 세척전,2025-06-12,35.273232,126.512825,전라남도,영광군
676525,1,O000084,"정액,수정란,종란",기타,팽이버섯 3,2025-10-20,35.132931,127.254112,전라남도,곡성군
676526,0,O000858,기타기준규격외,기타기준규격외,별별생선까스 동결 후,2025-05-19,37.385250,127.287964,경기도,광주시
676528,0,O000169,기타기준규격외,기타기준규격외,편육 냉각,2025-06-16,37.744717,126.805168,경기도,파주시


In [ ]:
tf

,judge_result,inspct_instt_code,train_prdlst_nm,train_prdlst_nm_2,prduct_nm,mnfctur_de,lat,lon,sido,sigungu
0,0,O002012,즉석식품류,즉석섭취.편의식품류,렌위치 NY 샌드위치,2025-10-11,36.975372,127.215195,경기도,안성시
1,0,O000728,기타기준규격외,기타기준규격외,"삼치(국내산_해동,전처리,세척전)",2025-04-28,34.708126,127.613495,전라남도,여수시
5,0,O000473,기타기준규격외,기타기준규격외,돈육(냉동)(국내)절단/성형후1차,2025-10-15,37.823318,127.138603,경기도,포천시
7,0,O000392,기타기준규격외,기타기준규격외,표고버섯분말 살균후 (완제품),2025-05-23,36.444393,126.832138,충청남도,청양군
9,0,O000858,기타기준규격외,기타기준규격외,갑오징어-원재료,2025-08-25,35.991085,126.689377,전북특별자치도,군산시
...,...,...,...,...,...,...,...,...,...,...
676524,0,O000728,기타기준규격외,기타기준규격외,쌀 세척전,2025-06-12,35.273232,126.512825,전라남도,영광군
676525,1,O000084,"정액,수정란,종란",기타,팽이버섯 3,2025-10-20,35.132931,127.254112,전라남도,곡성군
676526,0,O000858,기타기준규격외,기타기준규격외,별별생선까스 동결 후,2025-05-19,37.385250,127.287964,경기도,광주시
676528,0,O000169,기타기준규격외,기타기준규격외,편육 냉각,2025-06-16,37.744717,126.805168,경기도,파주시


### **3. 품목군별 Target Mean 적용**
- 데이터가 충분히 많기 떄문에 과거 부적합 비율을 사용해도 무방하다고 판단(Data Leakage는 없다고 판단)

In [ ]:
# train_prdlst_nm_2 단위로 judge_result의 평균값 계산하여 새로운 컬럼으로 추가
target_means = tf.groupby('train_prdlst_nm_2')['judge_result'].mean()
tf['target_mean_by_product_2'] = tf['train_prdlst_nm_2'].map(target_means)
tf[['train_prdlst_nm_2', 'prduct_nm', 'target_mean_by_product_2']].head()

/tmp/ipykernel_1213/53235399.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  target_means = tf.groupby('train_prdlst_nm_2')['judge_result'].mean()


,train_prdlst_nm_2,prduct_nm,target_mean_by_product_2
0,즉석섭취.편의식품류,렌위치 NY 샌드위치,0.001179
1,기타기준규격외,"삼치(국내산_해동,전처리,세척전)",0.002711
5,기타기준규격외,돈육(냉동)(국내)절단/성형후1차,0.002711
7,기타기준규격외,표고버섯분말 살균후 (완제품),0.002711
9,기타기준규격외,갑오징어-원재료,0.002711


### **3. 대한 민국을 pixel로 만들어 지역정보를 구분(위도 경도 정보 --> 격자 Grid)**
- 대한민국 정도의 좁은 지역에서는 위도·경도를 다음처럼 근사

> 위도 1° ≈ 111 km

> 경도 1° ≈ 88~92 km 정도

- **make_grid() 함수로 km 단위로 격자 셀을 생성하여 시계열정보와 연계하여 예측 가능**

> **10km, 30km, 50km 등등 Hyperparameter Tunning 실행**

> **기후정보와 join시 위도와 경도가 같은 grid 이면 Join하여 사용**


In [ ]:
def make_grid(df, km):
    gdf = gpd.GeoDataFrame(
        df.copy(),
        geometry=gpd.points_from_xy(df['Lon'], df['Lat']),
        crs='EPSG:4326'
    )

    gdf = gdf.to_crs('EPSG:5179')

    grid_size = km * 1000

    gdf['grid_x'] = (gdf.geometry.x // grid_size).astype(int)
    gdf['grid_y'] = (gdf.geometry.y // grid_size).astype(int)

    gdf['Grid_ID'] = (
        gdf['grid_x'].astype(str) + '_' +
        gdf['grid_y'].astype(str)
    )

    return gdf

In [ ]:
import geopandas as gpd

# Lon, Lat을 소문자 lon, lat 컬럼명에 맞춰 적용되도록 make_grid 함수 재정의 및 실행
def make_grid_fixed(df, km):
    gdf = gpd.GeoDataFrame(
        df.copy(),
        geometry=gpd.points_from_xy(df['lon'], df['lat']),
        crs='EPSG:4326'
    )
    gdf = gdf.to_crs('EPSG:5179')
    grid_size = km * 1000
    gdf['grid_x'] = (gdf.geometry.x // grid_size).astype(int)
    gdf['grid_y'] = (gdf.geometry.y // grid_size).astype(int)
    gdf['Grid_ID'] = (
        gdf['grid_x'].astype(str) + '_'.encode().decode() +
        gdf['grid_y'].astype(str)
    )
    return gdf


In [ ]:
# @title 10 km × 10 km 크기의 격자 중에서 97번째 × 188번째 위치
tf_grid = make_grid_fixed(tf, 10) # make_grid_fixed(tf, 20)
tf_grid.head()

,judge_result,inspct_instt_code,train_prdlst_nm,train_prdlst_nm_2,prduct_nm,mnfctur_de,lat,lon,sido,sigungu,target_mean_by_product_2,geometry,grid_x,grid_y,Grid_ID
0,0,O002012,즉석식품류,즉석섭취.편의식품류,렌위치 NY 샌드위치,2025-10-11,36.975372,127.215195,경기도,안성시,0.001179,POINT (974650.945 1886363.221),97,188,97_188
1,0,O000728,기타기준규격외,기타기준규격외,"삼치(국내산_해동,전처리,세척전)",2025-04-28,34.708126,127.613495,전라남도,여수시,0.002711,POINT (1010393.238 1634866.975),101,163,101_163
5,0,O000473,기타기준규격외,기타기준규격외,돈육(냉동)(국내)절단/성형후1차,2025-10-15,37.823318,127.138603,경기도,포천시,0.002711,POINT (968194.301 1980458.626),96,198,96_198
7,0,O000392,기타기준규격외,기타기준규격외,표고버섯분말 살균후 (완제품),2025-05-23,36.444393,126.832138,충청남도,청양군,0.002711,POINT (940146.307 1827632.194),94,182,94_182
9,0,O000858,기타기준규격외,기타기준규격외,갑오징어-원재료,2025-08-25,35.991085,126.689377,전북특별자치도,군산시,0.002711,POINT (926931.478 1777448.402),92,177,92_177


In [ ]:
# drop() 메소드를 사용하여 여러 컬럼을 한 번에 삭제합니다.
tf_grid = tf_grid.drop(columns=['Grid_ID', 'geometry'])
tf_grid.head()

,judge_result,inspct_instt_code,train_prdlst_nm,train_prdlst_nm_2,prduct_nm,mnfctur_de,lat,lon,sido,sigungu,target_mean_by_product_2,grid_x,grid_y
0,0,O002012,즉석식품류,즉석섭취.편의식품류,렌위치 NY 샌드위치,2025-10-11,36.975372,127.215195,경기도,안성시,0.001179,97,188
1,0,O000728,기타기준규격외,기타기준규격외,"삼치(국내산_해동,전처리,세척전)",2025-04-28,34.708126,127.613495,전라남도,여수시,0.002711,101,163
5,0,O000473,기타기준규격외,기타기준규격외,돈육(냉동)(국내)절단/성형후1차,2025-10-15,37.823318,127.138603,경기도,포천시,0.002711,96,198
7,0,O000392,기타기준규격외,기타기준규격외,표고버섯분말 살균후 (완제품),2025-05-23,36.444393,126.832138,충청남도,청양군,0.002711,94,182
9,0,O000858,기타기준규격외,기타기준규격외,갑오징어-원재료,2025-08-25,35.991085,126.689377,전북특별자치도,군산시,0.002711,92,177


In [ ]:
tf_grid

,judge_result,inspct_instt_code,train_prdlst_nm,train_prdlst_nm_2,prduct_nm,mnfctur_de,lat,lon,sido,sigungu,target_mean_by_product_2,grid_x,grid_y
0,0,O002012,즉석식품류,즉석섭취.편의식품류,렌위치 NY 샌드위치,2025-10-11,36.975372,127.215195,경기도,안성시,0.001179,97,188
1,0,O000728,기타기준규격외,기타기준규격외,"삼치(국내산_해동,전처리,세척전)",2025-04-28,34.708126,127.613495,전라남도,여수시,0.002711,101,163
5,0,O000473,기타기준규격외,기타기준규격외,돈육(냉동)(국내)절단/성형후1차,2025-10-15,37.823318,127.138603,경기도,포천시,0.002711,96,198
7,0,O000392,기타기준규격외,기타기준규격외,표고버섯분말 살균후 (완제품),2025-05-23,36.444393,126.832138,충청남도,청양군,0.002711,94,182
9,0,O000858,기타기준규격외,기타기준규격외,갑오징어-원재료,2025-08-25,35.991085,126.689377,전북특별자치도,군산시,0.002711,92,177
...,...,...,...,...,...,...,...,...,...,...,...,...,...
676524,0,O000728,기타기준규격외,기타기준규격외,쌀 세척전,2025-06-12,35.273232,126.512825,전라남도,영광군,0.002711,91,169
676525,1,O000084,"정액,수정란,종란",기타,팽이버섯 3,2025-10-20,35.132931,127.254112,전라남도,곡성군,0.022236,97,168
676526,0,O000858,기타기준규격외,기타기준규격외,별별생선까스 동결 후,2025-05-19,37.385250,127.287964,경기도,광주시,0.002711,98,193
676528,0,O000169,기타기준규격외,기타기준규격외,편육 냉각,2025-06-16,37.744717,126.805168,경기도,파주시,0.002711,93,197


### **4. 제품명에 대한 text mining으로 feature 생성**
- LLM에게 제품명에 대해 부적합비율과 관련성이 높은 text를 clustering prompt

In [ ]:
tf_grid[['train_prdlst_nm', 'train_prdlst_nm_2', 'prduct_nm', 'target_mean_by_product_2']]

,train_prdlst_nm,train_prdlst_nm_2,prduct_nm,target_mean_by_product_2
0,즉석식품류,즉석섭취.편의식품류,렌위치 NY 샌드위치,0.001179
1,기타기준규격외,기타기준규격외,"삼치(국내산_해동,전처리,세척전)",0.002711
5,기타기준규격외,기타기준규격외,돈육(냉동)(국내)절단/성형후1차,0.002711
7,기타기준규격외,기타기준규격외,표고버섯분말 살균후 (완제품),0.002711
9,기타기준규격외,기타기준규격외,갑오징어-원재료,0.002711
...,...,...,...,...
676524,기타기준규격외,기타기준규격외,쌀 세척전,0.002711
676525,"정액,수정란,종란",기타,팽이버섯 3,0.022236
676526,기타기준규격외,기타기준규격외,별별생선까스 동결 후,0.002711
676528,기타기준규격외,기타기준규격외,편육 냉각,0.002711


In [ ]:
df = tf_grid.copy()

In [ ]:
import re

# 2. 리스테리아 특화 파생 변수 추출 함수
def extract_listeria_features(text):
    text_str = str(text).strip()
    text_lower = text_str.lower()

    # --- A. 가열 / 살균 상태 (heat_process) ---
    heat_process = '기타/공정중'
    if re.search(r'살균후|가열후|구이후|볶음후|증자후|멸균|훈연|훈제', text_str):
        heat_process = '가열_살균후'
    elif re.search(
        r'살균전|가열전|구이전|생식|비가열|세척후|전처리후|원재료|절단', text_str
    ):
        heat_process = '가열전_비가열_원재료'
    elif re.search(r'ccp', text_lower):
        heat_process = 'CCP공정'

    # --- B. 공정 및 완제품 단계 (process_stage) ---
    process_stage = '공정중'
    if re.search(
        r'완제품|포장후|바로섭취|도시락|샌드위치|김밥|샐러드|마들렌|프레첼|강정',
        text_str,
    ):
        process_stage = '완제품_RTE가능'
    elif re.search(r'원재료|원료|세척|전처리|절단|성형', text_str):
        process_stage = '원재료_전처리'

    # --- C. 보관 및 제형 특성 (storage_form) ---
    storage_form = '상온/일반'
    if re.search(r'냉장|슬라이스|해동|훈제|훈연|습윤', text_str):
        storage_form = '냉장_습윤'
    elif re.search(r'냉동', text_str):
        storage_form = '냉동'
    elif re.search(r'동결건조|건조|분말|가루|입자|멸균', text_str):
        storage_form = '건조_분말_멸균'

    # --- D. 리스테리아 고위험 개별 지표 (Binary) ---
    # RTE(즉석섭취)/훈제/유제품/가공육/샐러드/샌드위치 등 리스테리아 고위험 키워드
    is_rte_risk = int(
        bool(
            re.search(
                r'샌드위치|김밥|샐러드|훈제|훈연|슬라이스|치즈|유제품|참치마요|도시락|즉석|바로|완제품',
                text_str,
            )
        )
    )

    # 냉장/해동/세척후 등 습윤 상태
    is_chilled_wet = int(
        bool(re.search(r'냉장|해동|세척후|생식|습윤', text_str))
    )

    # --- E. 교차 변수 (Interaction Terms) ---
    # 1) 가열/살균 상태 x 공정 단계
    heat_X_stage = f'{heat_process}_X_{process_stage}'

    # 2) 가열/살균 상태 x 보관/제형 특성
    heat_X_storage = f'{heat_process}_X_{storage_form}'

    # 3) RTE 고위험 여부 x 냉장/습윤 여부 (1이면 최우선 리스테리아 고위험군)
    rte_X_chilled = is_rte_risk * is_chilled_wet

    return pd.Series([
        heat_process,
        process_stage,
        storage_form,
        is_rte_risk,
        is_chilled_wet,
        heat_X_stage,
        heat_X_storage,
        rte_X_chilled,
    ])


# 3. 데이터프레임 변수 적용
feature_cols = [
    'heat_process',
    'process_stage',
    'storage_form',
    'is_rte_risk',
    'is_chilled_wet',
    'heat_X_stage',
    'heat_X_storage',
    'rte_X_chilled',
]

In [ ]:
df[feature_cols] = df['prduct_nm'].apply(extract_listeria_features)
df

,judge_result,inspct_instt_code,train_prdlst_nm,train_prdlst_nm_2,prduct_nm,mnfctur_de,lat,lon,sido,sigungu,...,grid_x,grid_y,heat_process,process_stage,storage_form,is_rte_risk,is_chilled_wet,heat_X_stage,heat_X_storage,rte_X_chilled
0,0,O002012,즉석식품류,즉석섭취.편의식품류,렌위치 NY 샌드위치,2025-10-11,36.975372,127.215195,경기도,안성시,...,97,188,기타/공정중,완제품_RTE가능,상온/일반,1,0,기타/공정중_X_완제품_RTE가능,기타/공정중_X_상온/일반,0
1,0,O000728,기타기준규격외,기타기준규격외,"삼치(국내산_해동,전처리,세척전)",2025-04-28,34.708126,127.613495,전라남도,여수시,...,101,163,기타/공정중,원재료_전처리,냉장_습윤,0,1,기타/공정중_X_원재료_전처리,기타/공정중_X_냉장_습윤,0
5,0,O000473,기타기준규격외,기타기준규격외,돈육(냉동)(국내)절단/성형후1차,2025-10-15,37.823318,127.138603,경기도,포천시,...,96,198,가열전_비가열_원재료,원재료_전처리,냉동,0,0,가열전_비가열_원재료_X_원재료_전처리,가열전_비가열_원재료_X_냉동,0
7,0,O000392,기타기준규격외,기타기준규격외,표고버섯분말 살균후 (완제품),2025-05-23,36.444393,126.832138,충청남도,청양군,...,94,182,가열_살균후,완제품_RTE가능,건조_분말_멸균,1,0,가열_살균후_X_완제품_RTE가능,가열_살균후_X_건조_분말_멸균,0
9,0,O000858,기타기준규격외,기타기준규격외,갑오징어-원재료,2025-08-25,35.991085,126.689377,전북특별자치도,군산시,...,92,177,가열전_비가열_원재료,원재료_전처리,상온/일반,0,0,가열전_비가열_원재료_X_원재료_전처리,가열전_비가열_원재료_X_상온/일반,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
676524,0,O000728,기타기준규격외,기타기준규격외,쌀 세척전,2025-06-12,35.273232,126.512825,전라남도,영광군,...,91,169,기타/공정중,원재료_전처리,상온/일반,0,0,기타/공정중_X_원재료_전처리,기타/공정중_X_상온/일반,0
676525,1,O000084,"정액,수정란,종란",기타,팽이버섯 3,2025-10-20,35.132931,127.254112,전라남도,곡성군,...,97,168,기타/공정중,공정중,상온/일반,0,0,기타/공정중_X_공정중,기타/공정중_X_상온/일반,0
676526,0,O000858,기타기준규격외,기타기준규격외,별별생선까스 동결 후,2025-05-19,37.385250,127.287964,경기도,광주시,...,98,193,기타/공정중,공정중,상온/일반,0,0,기타/공정중_X_공정중,기타/공정중_X_상온/일반,0
676528,0,O000169,기타기준규격외,기타기준규격외,편육 냉각,2025-06-16,37.744717,126.805168,경기도,파주시,...,93,197,기타/공정중,공정중,상온/일반,0,0,기타/공정중_X_공정중,기타/공정중_X_상온/일반,0


### **5. prduct_nm의 전체 텍스트에 대해 Char/Word N-gram 기반 TF-IDF Vectorizer를 적용**
- TF-IDF + 머신러닝/딥러닝 연계: TF-IDF Vectorizer를 적용한 후 LightGBM, XGBoost 또는 TabNet 모델에 함께 입력값으로 넣어주면 텍스트의 정교한 패턴까지 학습에 활용

In [ ]:
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix, hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    f1_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# 모델 불러오기
import lightgbm as lgb
from xgboost import XGBClassifier

# ---------------------------------------------------------
# 1. 타겟 변수(y) 및 입력 데이터(X) 준비
# ---------------------------------------------------------
TARGET_COL = 'judge_result'

# Train / Test Split (Stratify 적용)
X_train_df, X_test_df, y_train, y_test = train_test_split(
    df, df[TARGET_COL], test_size=0.2, random_state=42, stratify=df[TARGET_COL]
)

# ---------------------------------------------------------
# 2. TF-IDF 피처 추출 (Char N-gram + Word N-gram)
# ---------------------------------------------------------
# 결측치(Null/NA) 처리: 결측치를 빈 문자열으로 변환하여 TfidfVectorizer의 NAType 오류 방지
X_train_text = X_train_df['prduct_nm'].fillna('').astype(str)
X_test_text = X_test_df['prduct_nm'].fillna('').astype(str)

# Char N-gram: 글자 단위 (2~4자 연쇄) -> 띄어쓰기 오차, 붙여쓴 단어에 강함
tfidf_char = TfidfVectorizer(
    analyzer='char', ngram_range=(2, 4), max_features=300, min_df=5
)

# Word N-gram: 단어 단위 (1~2단어 조합)
tfidf_word = TfidfVectorizer(
    analyzer='word', ngram_range=(1, 2), max_features=200, min_df=3
)

# Fit & Transform
X_train_char = tfidf_char.fit_transform(X_train_text)
X_train_word = tfidf_word.fit_transform(X_train_text)

X_test_char = tfidf_char.transform(X_test_text)
X_test_word = tfidf_word.transform(X_test_text)

# ---------------------------------------------------------
# 3. 규칙 기반(Rule-based) 생성 변수 인코딩 및 스케일링
# ---------------------------------------------------------
cat_cols = [
    'heat_process',
    'process_stage',
    'storage_form',
    'heat_X_stage',
    'heat_X_storage',
]
num_cols = ['is_rte_risk', 'is_chilled_wet', 'rte_X_chilled']

# 범주형 -> One-Hot Encoding
ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=True)
X_train_cat = ohe.fit_transform(X_train_df[cat_cols])
X_test_cat = ohe.transform(X_test_df[cat_cols])

# 수치형 -> StandardScaler
scaler = StandardScaler()
X_train_num = scaler.fit_transform(X_train_df[num_cols])
X_test_num = scaler.transform(X_test_df[num_cols])

# ---------------------------------------------------------
# 4. 희소 행렬(Sparse Matrix) 하나로 결합
# ---------------------------------------------------------
# [규칙 범주형(OHE) + 규칙 수치형 + TF-IDF Char + TF-IDF Word]
X_train_final = hstack([
    X_train_cat,
    csr_matrix(X_train_num),
    X_train_char,
    X_train_word,
]).tocsr()

X_test_final = hstack([
    X_test_cat,
    csr_matrix(X_test_num),
    X_test_char,
    X_test_word,
]).tocsr()

print(f'최종 Train 피처 수: {X_train_final.shape[1]}개')
print(f'최종 Test 피처 수: {X_test_final.shape[1]}개')


최종 Train 피처 수: 542개
최종 Test 피처 수: 542개


### **6. 전체 모델 평가**

In [ ]:
!pip install catboost -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 9.0 MB/s eta 0:00:00


In [ ]:
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix, hstack
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    auc,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Gradient Boosting 계열 모델
import lightgbm as lgb
from catboost import CatBoostClassifier
from xgboost import XGBClassifier

In [ ]:
# ---------------------------------------------------------
# 3. 주요 모델 정의 (불균형 데이터 보정 포함)
# ---------------------------------------------------------
ratio = (len(y_train) - sum(y_train)) / sum(y_train)

models = {
    'LightGBM': lgb.LGBMClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=7,
        class_weight='balanced',
        random_state=42,
        n_jobs=-1,
    ),
    'XGBoost': XGBClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=6,
        scale_pos_weight=ratio,
        random_state=42,
        n_jobs=-1,
        eval_metric='logloss',
    ),
    'CatBoost': CatBoostClassifier(
        iterations=300,
        learning_rate=0.05,
        scale_pos_weight=ratio,
        random_state=42,
        verbose=0,
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=150, class_weight='balanced', random_state=42, n_jobs=-1
    ),
    'Logistic Regression': LogisticRegression(
        class_weight='balanced', max_iter=1000, random_state=42
    ),
}

# ---------------------------------------------------------
# 4. 모델별 학습, 평가 및 오탐/PR-AUC 지표 계산
# ---------------------------------------------------------
results = []

for name, model in models.items():
    print(f'\n=== {name} 모델 학습 및 평가 시작 ===')

    # 학습
    model.fit(X_train_final, y_train)

    # 예측 (확률 및 클래스)
    pred_prob = model.predict_proba(X_test_final)[:, 1]
    pred = (pred_prob >= 0.5).astype(int)

    # 1) PR-AUC 계산
    precision_vec, recall_vec, _ = precision_recall_curve(y_test, pred_prob)
    pr_auc_val = auc(recall_vec, precision_vec)

    # 2) Confusion Matrix 오탐지표 계산 (FN, FP)
    tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
    total_errors = fn + fp
    error_rate = (total_errors / len(y_test)) * 100

    # 3) 기타 평가 지표
    roc_auc_val = roc_auc_score(y_test, pred_prob)
    f1_val = f1_score(y_test, pred, zero_division=0)
    rec_val = recall_score(y_test, pred, zero_division=0)
    prec_val = precision_score(y_test, pred, zero_division=0)

    print(f'[{name} 평가 결과]')
    print(f'PR-AUC Score        : {pr_auc_val:.4f}')
    print(f'ROC-AUC Score       : {roc_auc_val:.4f}')
    print(f'F1-Score            : {f1_val:.4f}')
    print(
        f'총 오류 건수(FN+FP)  : {total_errors:,}건 (미검출 FN: {fn}건, 과잉검출 FP: {fp}건)'
    )
    print(f'오탐 비율(%)        : {error_rate:.2f}%')
    print('\n[Classification Report]')
    print(classification_report(y_test, pred, zero_division=0))

    results.append({
        'Model': name,
        'PR-AUC': pr_auc_val,
        '오류건수(FN+FP)': total_errors,
        '오탐비율(%)': error_rate,
        '미검출(FN)': fn,
        '과잉검출(FP)': fp,
        'ROC-AUC': roc_auc_val,
        'F1-Score': f1_val,
        'Recall': rec_val,
        'Precision': prec_val,
    })

# ---------------------------------------------------------
# 5. 모델 비교 요약 표 출력
# ---------------------------------------------------------
summary_df = (
    pd.DataFrame(results)
    .sort_values(by='PR-AUC', ascending=False)
    .reset_index(drop=True)
)

print('\n' + '=' * 60)
print('[전체 모델 종합 성능 비교 요약 (PR-AUC 순 정렬)]')
print('=' * 60)
print(summary_df.to_string(index=False))

In [ ]:
summary_df

,Model,PR-AUC,오류건수(FN+FP),오탐비율(%),미검출(FN),과잉검출(FP),ROC-AUC,F1-Score,Recall,Precision
0,XGBoost,0.108430,9041,9.053584,178,8863,0.809543,0.040539,0.517615,0.021096
1,LightGBM,0.082013,10056,10.069997,171,9885,0.810574,0.037887,0.536585,0.019637
2,Random Forest,0.070780,5801,5.809075,232,5569,0.706736,0.045103,0.371274,0.024010
3,CatBoost,0.058392,14227,14.246803,155,14072,0.811771,0.029205,0.579946,0.014980
4,Logistic Regression,0.045384,18890,18.916294,138,18752,0.810179,0.023874,0.626016,0.012169


### **7. 데이터 특성 번영 전체 모델 평가(휴리스틱 수동 설정)**
- 불균형 가중치 조정: 전체 ratio를 그대로 쓰면 과잉검출(FP)이 수천 건 급증하므로, scale_pos_weight 지수를 완화 조율
- 모델 복잡도 제한: max_depth, num_leaves, min_child_samples, min_samples_leaf 등을 통해 트리 깊이를 제한하여 FP 발생을 억제
- F1-Score 기반 임계값(Threshold) 자동 탐색: 고정 $0.5$ 임계값 대신 Precision-Recall 커브에서 F1-Score가 최대가 되는 최적의 임계값을 찾아 분류 기준으로 적용

In [ ]:
# ---------------------------------------------------------
# 1. 불균형 가중치 및 기본 설정
# ---------------------------------------------------------
# 양성/음성 비율 계산
num_pos = y_train.sum()
num_neg = len(y_train) - num_pos

# 기존 극단적 ratio 대신 가중치를 완화하여 과잉검출(FP)을 억제
# (기본 ratio의 0.2~0.3 수준으로 축소 설정하여 Precision 개선)
balanced_ratio = (num_neg / num_pos) * 0.25

# ---------------------------------------------------------
# 2. 주요 모델 정의 (가중치 완화 + 복잡도 제어 적용)
# ---------------------------------------------------------
models = {
    'LightGBM': lgb.LGBMClassifier(
        n_estimators=200,
        learning_rate=0.03,
        max_depth=5,  # 복잡도 제어
        num_leaves=20,  # 과적합 방지
        min_child_samples=30,  # 노드 최소 데이터 수 증가
        scale_pos_weight=balanced_ratio,
        random_state=42,
        n_jobs=-1,
    ),
    'XGBoost': XGBClassifier(
        n_estimators=200,
        learning_rate=0.03,
        max_depth=4,  # 복잡도 제어 (FP 방지)
        min_child_weight=5,  # 과적합 방지
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=balanced_ratio,
        random_state=42,
        n_jobs=-1,
        eval_metric='logloss',
    ),
    'CatBoost': CatBoostClassifier(
        iterations=200,
        learning_rate=0.03,
        depth=5,  # 복잡도 제어
        scale_pos_weight=balanced_ratio,
        random_state=42,
        verbose=0,
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=150,
        max_depth=8,  # 복잡도 제어
        min_samples_leaf=10,  # 과적합 방지
        class_weight={0: 1, 1: 5},  # 가중치 완화 수동 지정
        random_state=42,
        n_jobs=-1,
    ),
    'Logistic Regression': LogisticRegression(
        class_weight={0: 1, 1: 5},  # 가중치 완화 수동 지정
        C=0.1,  # 규제 강화 (과적합 방지)
        max_iter=1000,
        random_state=42,
    ),
}

# ---------------------------------------------------------
# 3. 모델별 학습, F1 최적화 임계값 적용 및 지표 계산
# ---------------------------------------------------------
results = []

for name, model in models.items():
    print(f'\n=== {name} 모델 학습 및 평가 시작 ===')

    # 학습
    model.fit(X_train_final, y_train)

    # 예측 확률 (Positive Class)
    pred_prob = model.predict_proba(X_test_final)[:, 1]

    # PR 커브 추출
    precision_vec, recall_vec, thresholds = precision_recall_curve(
        y_test, pred_prob
    )

    # 1) F1-Score 기준 최적 Decision Threshold 찾기
    f1_scores = np.where(
        (precision_vec + recall_vec) == 0,
        0,
        2 * (precision_vec * recall_vec) / (precision_vec + recall_vec),
    )
    best_idx = np.argmax(f1_scores)

    # thresholds 배열 길이가 1 짧으므로 인덱스 범위 지정
    best_threshold = (
        thresholds[best_idx] if best_idx < len(thresholds) else 0.5
    )

    # 최적 임계값 적용하여 예측 클래스 생성
    pred = (pred_prob >= best_threshold).astype(int)

    # 2) 주요 평가 지표 계산
    pr_auc_val = auc(recall_vec, precision_vec)
    roc_auc_val = roc_auc_score(y_test, pred_prob)

    tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
    total_errors = fn + fp
    error_rate = (total_errors / len(y_test)) * 100

    f1_val = f1_score(y_test, pred, zero_division=0)
    rec_val = recall_score(y_test, pred, zero_division=0)
    prec_val = precision_score(y_test, pred, zero_division=0)

    print(f'[{name} 평가 결과 (적용된 최적 Threshold: {best_threshold:.4f})]')
    print(f'F1-Score            : {f1_val:.4f}')
    print(f'PR-AUC Score        : {pr_auc_val:.4f}')
    print(f'ROC-AUC Score       : {roc_auc_val:.4f}')
    print(f'Precision           : {prec_val:.4f}')
    print(f'Recall              : {rec_val:.4f}')
    print(
        f'총 오류 건수(FN+FP)  : {total_errors:,}건 (미검출 FN: {fn}건, 과잉검출 FP:'
        f' {fp}건)'
    )
    print(f'오탐 비율(%)        : {error_rate:.2f}%')
    print('\n[Classification Report]')
    print(classification_report(y_test, pred, zero_division=0))

    results.append({
        'Model': name,
        'Best Threshold': round(best_threshold, 4),
        'F1-Score': f1_val,
        'PR-AUC': pr_auc_val,
        'ROC-AUC': roc_auc_val,
        'Precision': prec_val,
        'Recall': rec_val,
        '오류건수(FN+FP)': total_errors,
        '오탐비율(%)': error_rate,
        '미검출(FN)': fn,
        '과잉검출(FP)': fp,
    })

# ---------------------------------------------------------
# 4. F1-Score 기준 최종 비교 요약 표 출력
# ---------------------------------------------------------
summary_df = (
    pd.DataFrame(results)
    .sort_values(by='F1-Score', ascending=False)
    .reset_index(drop=True)
)

print('\n' + '=' * 70)
print('[전체 모델 종합 성능 비교 요약 (F1-Score 순 정렬)]')
print('=' * 70)
print(summary_df.to_string(index=False))


=== LightGBM 모델 학습 및 평가 시작 ===
[LightGBM] [Info] Number of positive: 1475, number of negative: 397967
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 1.557409 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 80368
[LightGBM] [Info] Number of data points in the train set: 399442, number of used features: 542
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.003693 -> initscore=-5.597711
[LightGBM] [Info] Start training from score -5.597711
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further 

/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM 평가 결과 (적용된 최적 Threshold: 0.9307)]
F1-Score            : 0.1773
PR-AUC Score        : 0.0760
ROC-AUC Score       : 0.7891
Precision           : 0.1572
Recall              : 0.2033
총 오류 건수(FN+FP)  : 696건 (미검출 FN: 294건, 과잉검출 FP: 402건)
오탐 비율(%)        : 0.70%

[Classification Report]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     99492
           1       0.16      0.20      0.18       369

    accuracy                           0.99     99861
   macro avg       0.58      0.60      0.59     99861
weighted avg       0.99      0.99      0.99     99861


=== XGBoost 모델 학습 및 평가 시작 ===
[XGBoost 평가 결과 (적용된 최적 Threshold: 0.7545)]
F1-Score            : 0.1861
PR-AUC Score        : 0.0784
ROC-AUC Score       : 0.7871
Precision           : 0.1537
Recall              : 0.2358
총 오류 건수(FN+FP)  : 761건 (미검출 FN: 282건, 과잉검출 FP: 479건)
오탐 비율(%)        : 0.76%

[Classification Report]
              precision    recall  f1-score   support

      

/tmp/ipykernel_1071/60234967.py:85: RuntimeWarning: invalid value encountered in divide
  2 * (precision_vec * recall_vec) / (precision_vec + recall_vec),


[CatBoost 평가 결과 (적용된 최적 Threshold: 0.7783)]
F1-Score            : 0.1772
PR-AUC Score        : 0.0519
ROC-AUC Score       : 0.7958
Precision           : 0.1604
Recall              : 0.1978
총 오류 건수(FN+FP)  : 678건 (미검출 FN: 296건, 과잉검출 FP: 382건)
오탐 비율(%)        : 0.68%

[Classification Report]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     99492
           1       0.16      0.20      0.18       369

    accuracy                           0.99     99861
   macro avg       0.58      0.60      0.59     99861
weighted avg       0.99      0.99      0.99     99861


=== Random Forest 모델 학습 및 평가 시작 ===
[Random Forest 평가 결과 (적용된 최적 Threshold: 0.2497)]
F1-Score            : 0.1881
PR-AUC Score        : 0.0807
ROC-AUC Score       : 0.7641
Precision           : 0.1794
Recall              : 0.1978
총 오류 건수(FN+FP)  : 630건 (미검출 FN: 296건, 과잉검출 FP: 334건)
오탐 비율(%)        : 0.63%

[Classification Report]
              precision    recall  f1-score   sup

/tmp/ipykernel_1071/60234967.py:85: RuntimeWarning: invalid value encountered in divide
  2 * (precision_vec * recall_vec) / (precision_vec + recall_vec),


In [ ]:
summary_df

,Model,Best Threshold,F1-Score,PR-AUC,ROC-AUC,Precision,Recall,오류건수(FN+FP),오탐비율(%),미검출(FN),과잉검출(FP)
0,Random Forest,0.2497,0.188144,0.080671,0.764062,0.179361,0.197832,630,0.630877,296,334
1,XGBoost,0.7545,0.186096,0.078447,0.787063,0.153710,0.235772,761,0.762059,282,479
2,LightGBM,0.9307,0.177305,0.076040,0.789087,0.157233,0.203252,696,0.696969,294,402
3,CatBoost,0.7783,0.177184,0.051902,0.795844,0.160440,0.197832,678,0.678944,296,382
4,Logistic Regression,0.3038,0.172414,0.057148,0.807157,0.168831,0.176152,624,0.624869,304,320


### **8. Optuna 튜닝 진행**

In [ ]:
import warnings
# scikit-learn 및 LightGBM 관련 UserWarning 안 보이게 설정
warnings.filterwarnings('ignore', category=UserWarning)

In [ ]:
!pip install optuna -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 442.4/442.4 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.7/268.7 kB 10.4 MB/s eta 0:00:00


In [ ]:
import optuna
import time

start = time.perf_counter()

# Optuna 로그 출력 최소화
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ---------------------------------------------------------
# 1. Optuna 목적 함수 (Objective Functions) 정의
# ---------------------------------------------------------
# 1) LightGBM Objective
def objective_lgb(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 300),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 8),
        'num_leaves': trial.suggest_int('num_leaves', 15, 63),
        'min_child_samples': trial.suggest_int('min_child_samples', 20, 100),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'scale_pos_weight': trial.suggest_float('scale_pos_weight', 1.0, 50.0, log=True),
        'random_state': 42,
        'n_jobs': -1,
        'verbose': -1
    }

    model = lgb.LGBMClassifier(**params)
    model.fit(X_train_final, y_train)

    # Validation(Test) 예측 확률
    pred_prob = model.predict_proba(X_test_final)[:, 1]

    # F1-Score 최적화
    precision, recall, _ = precision_recall_curve(y_test, pred_prob)
    f1_scores = np.where((precision + recall) == 0, 0, 2 * (precision * recall) / (precision + recall))

    return np.max(f1_scores)

# 2) XGBoost Objective
def objective_xgb(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 300),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 8),
        'min_child_weight': trial.suggest_int('min_child_weight', 2, 20),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'scale_pos_weight': trial.suggest_float('scale_pos_weight', 1.0, 50.0, log=True),
        'random_state': 42,
        'n_jobs': -1,
        'eval_metric': 'logloss'
    }

    model = XGBClassifier(**params)
    model.fit(X_train_final, y_train)

    pred_prob = model.predict_proba(X_test_final)[:, 1]
    precision, recall, _ = precision_recall_curve(y_test, pred_prob)
    f1_scores = np.where((precision + recall) == 0, 0, 2 * (precision * recall) / (precision + recall))

    return np.max(f1_scores)

# 3) CatBoost Objective
def objective_cat(trial):
    params = {
        'iterations': trial.suggest_int('iterations', 100, 300),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'depth': trial.suggest_int('depth', 3, 8),
        'scale_pos_weight': trial.suggest_float('scale_pos_weight', 1.0, 50.0, log=True),
        'random_state': 42,
        'verbose': 0
    }

    model = CatBoostClassifier(**params)
    model.fit(X_train_final, y_train)

    pred_prob = model.predict_proba(X_test_final)[:, 1]
    precision, recall, _ = precision_recall_curve(y_test, pred_prob)
    f1_scores = np.where((precision + recall) == 0, 0, 2 * (precision * recall) / (precision + recall))

    return np.max(f1_scores)

# ---------------------------------------------------------
# 2. Optuna 튜닝 실행
# ---------------------------------------------------------
N_TRIALS = 30  # 시도 횟수 (필요 시 50~100으로 확대 가능)

print("=== LightGBM 튜닝 시작 ===")
study_lgb = optuna.create_study(direction='maximize')
study_lgb.optimize(objective_lgb, n_trials=N_TRIALS)

print("=== XGBoost 튜닝 시작 ===")
study_xgb = optuna.create_study(direction='maximize')
study_xgb.optimize(objective_xgb, n_trials=N_TRIALS)

print("=== CatBoost 튜닝 시작 ===")
study_cat = optuna.create_study(direction='maximize')
study_cat.optimize(objective_cat, n_trials=N_TRIALS)

# ---------------------------------------------------------
# 3. 최적 파라미터로 최종 모델 재학습 및 평가
# ---------------------------------------------------------
tuned_models = {
    'LightGBM': lgb.LGBMClassifier(**study_lgb.best_params, random_state=42, n_jobs=-1, verbose=-1),
    'XGBoost': XGBClassifier(**study_xgb.best_params, random_state=42, n_jobs=-1, eval_metric='logloss'),
    'CatBoost': CatBoostClassifier(**study_cat.best_params, random_state=42, verbose=0)
}

tuned_results = []

for name, model in tuned_models.items():
    print(f"\n=== {name} 최적 모델 평가 시작 ===")
    model.fit(X_train_final, y_train)

    pred_prob = model.predict_proba(X_test_final)[:, 1]

    # PR 커브 및 F1 최적 Threshold 탐색
    precision_vec, recall_vec, thresholds = precision_recall_curve(y_test, pred_prob)
    f1_scores = np.where((precision_vec + recall_vec) == 0, 0, 2 * (precision_vec * recall_vec) / (precision_vec + recall_vec))
    best_idx = np.argmax(f1_scores)
    best_threshold = thresholds[best_idx] if best_idx < len(thresholds) else 0.5

    # 최적 Threshold 적용
    pred = (pred_prob >= best_threshold).astype(int)

    # 평가 지표 산출
    pr_auc_val = auc(recall_vec, precision_vec)
    roc_auc_val = roc_auc_score(y_test, pred_prob)
    tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
    total_errors = fn + fp
    error_rate = (total_errors / len(y_test)) * 100

    f1_val = f1_score(y_test, pred, zero_division=0)
    rec_val = recall_score(y_test, pred, zero_division=0)
    prec_val = precision_score(y_test, pred, zero_division=0)

    print(f"[{name} 최적화 결과 (Best Threshold: {best_threshold:.4f})]")
    print(f"F1-Score            : {f1_val:.4f}")
    print(f"PR-AUC Score        : {pr_auc_val:.4f}")
    print(f"Precision           : {prec_val:.4f}")
    print(f"Recall              : {rec_val:.4f}")
    print(f"총 오류 건수(FN+FP)  : {total_errors:,}건 (미검출 FN: {fn}건, 과잉검출 FP: {fp}건)")

    tuned_results.append({
        'Model': name,
        'Best Threshold': round(best_threshold, 4),
        'F1-Score': f1_val,
        'PR-AUC': pr_auc_val,
        'ROC-AUC': roc_auc_val,
        'Precision': prec_val,
        'Recall': rec_val,
        '오류건수(FN+FP)': total_errors,
        '오탐비율(%)': error_rate,
        '미검출(FN)': fn,
        '과잉검출(FP)': fp
    })

# ---------------------------------------------------------
# 4. Optuna 튜닝 결과 요약 표 출력
# ---------------------------------------------------------
tuned_summary_df = pd.DataFrame(tuned_results).sort_values(by='F1-Score', ascending=False).reset_index(drop=True)

print('\n' + '='*70)
print('[Optuna 하이퍼파라미터 튜닝 최종 성능 비교 요약]')
print('='*70)
print(tuned_summary_df.to_string(index=False))



end = time.perf_counter()

print(f"실행 시간: {end - start:.4f}초")

=== LightGBM 튜닝 시작 ===


/tmp/ipykernel_1213/2429798240.py:36: RuntimeWarning: invalid value encountered in divide
  f1_scores = np.where((precision + recall) == 0, 0, 2 * (precision * recall) / (precision + recall))
/tmp/ipykernel_1213/2429798240.py:36: RuntimeWarning: invalid value encountered in divide
  f1_scores = np.where((precision + recall) == 0, 0, 2 * (precision * recall) / (precision + recall))
/tmp/ipykernel_1213/2429798240.py:36: RuntimeWarning: invalid value encountered in divide
  f1_scores = np.where((precision + recall) == 0, 0, 2 * (precision * recall) / (precision + recall))
/tmp/ipykernel_1213/2429798240.py:36: RuntimeWarning: invalid value encountered in divide
  f1_scores = np.where((precision + recall) == 0, 0, 2 * (precision * recall) / (precision + recall))


=== XGBoost 튜닝 시작 ===


/tmp/ipykernel_1213/2429798240.py:60: RuntimeWarning: invalid value encountered in divide
  f1_scores = np.where((precision + recall) == 0, 0, 2 * (precision * recall) / (precision + recall))
/tmp/ipykernel_1213/2429798240.py:60: RuntimeWarning: invalid value encountered in divide
  f1_scores = np.where((precision + recall) == 0, 0, 2 * (precision * recall) / (precision + recall))
/tmp/ipykernel_1213/2429798240.py:60: RuntimeWarning: invalid value encountered in divide
  f1_scores = np.where((precision + recall) == 0, 0, 2 * (precision * recall) / (precision + recall))
/tmp/ipykernel_1213/2429798240.py:60: RuntimeWarning: invalid value encountered in divide
  f1_scores = np.where((precision + recall) == 0, 0, 2 * (precision * recall) / (precision + recall))
/tmp/ipykernel_1213/2429798240.py:60: RuntimeWarning: invalid value encountered in divide
  f1_scores = np.where((precision + recall) == 0, 0, 2 * (precision * recall) / (precision + recall))
/tmp/ipykernel_1213/2429798240.py:60: Ru

=== CatBoost 튜닝 시작 ===


/tmp/ipykernel_1213/2429798240.py:80: RuntimeWarning: invalid value encountered in divide
  f1_scores = np.where((precision + recall) == 0, 0, 2 * (precision * recall) / (precision + recall))
/tmp/ipykernel_1213/2429798240.py:80: RuntimeWarning: invalid value encountered in divide
  f1_scores = np.where((precision + recall) == 0, 0, 2 * (precision * recall) / (precision + recall))
/tmp/ipykernel_1213/2429798240.py:80: RuntimeWarning: invalid value encountered in divide
  f1_scores = np.where((precision + recall) == 0, 0, 2 * (precision * recall) / (precision + recall))
/tmp/ipykernel_1213/2429798240.py:80: RuntimeWarning: invalid value encountered in divide
  f1_scores = np.where((precision + recall) == 0, 0, 2 * (precision * recall) / (precision + recall))
/tmp/ipykernel_1213/2429798240.py:80: RuntimeWarning: invalid value encountered in divide
  f1_scores = np.where((precision + recall) == 0, 0, 2 * (precision * recall) / (precision + recall))
/tmp/ipykernel_1213/2429798240.py:80: Ru

In [ ]:
tuned_summary_df

**Optuna 튜닝으로 개선된 핵심 원인**
- Precision과 Recall의 균형점 확보:휴리스틱에서는 Threshold를 0.9 이상으로 끌어올려 강제로 FP를 눌렀기 때문에 Recall이 떨어졌지만, Optuna가 scale_pos_weight와 트리 깊이를 동시에 최적화하면서 Recall(21~23%)을 유지한 채 Precision(20%)을 함께 끌어올렸습니다.   
- Best Threshold의 정상화:LightGBM 기준 Threshold가 0.93에서 0.14로 내려왔습니다. 이는 모델이 예측하는 확률값 분포 자체가 훨씬 신뢰할 수 있게 보정(Calibration)되었음을 의미합니다.   
- 오류 건수 감소:CatBoost의 경우 총 오류 건수가 678건에서 591건으로 줄어들며 가장 적은 오류를 기록했습니다.

### **9. Stacking Model after Optuna Tunned models**

- Optuna로 최적화된 3개 모델(LightGBM, XGBoost, CatBoost)의 Out-of-Fold(OOF) 예측 확률을 메타 피처(Meta-feature)로 생성하고, 이를 바탕으로 최종 Meta-Learner(Logistic Regression)를 학습시키는 스태킹(Stacking) 앙상블 코드

- 데이터 누수(Data Leakage)를 방지하기 위해 Stratified K-Fold 기반의 OOF 방식을 적용

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    f1_score, precision_recall_curve, auc, roc_auc_score,
    precision_score, recall_score, confusion_matrix, classification_report
)
import lightgbm as lgb
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

# ---------------------------------------------------------
# 1. Optuna 최적 파라미터로 베이스 모델 정의
# ---------------------------------------------------------
base_models = {
    'LightGBM': lgb.LGBMClassifier(**study_lgb.best_params, random_state=42, n_jobs=-1, verbose=-1),
    'XGBoost': XGBClassifier(**study_xgb.best_params, random_state=42, n_jobs=-1, eval_metric='logloss'),
    'CatBoost': CatBoostClassifier(**study_cat.best_params, random_state=42, verbose=0)
}

# ---------------------------------------------------------
# 2. Out-of-Fold(OOF) 기반 메타 피처(Meta-features) 생성
# ---------------------------------------------------------
N_SPLITS = 5
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

# 메타 피처 저장용 배열 초기화
oof_train = np.zeros((X_train_final.shape[0], len(base_models)))
oof_test = np.zeros((X_test_final.shape[0], len(base_models)))

print("=== Stacking용 OOF 메타 피처 생성 시작 ===")

for i, (name, model) in enumerate(base_models.items()):
    print(f"[{name}] K-Fold Cross Validation 진행 중...")

    test_preds_fold = np.zeros((X_test_final.shape[0], N_SPLITS))

    for fold, (train_idx, val_idx) in enumerate(skf.split(X_train_final, y_train)):
        X_tr, y_tr = X_train_final[train_idx], y_train.iloc[train_idx]
        X_va, y_va = X_train_final[val_idx], y_train.iloc[val_idx]

        # Fold별 모델 학습
        model.fit(X_tr, y_tr)

        # Validation 예측 확률 저장 (OOF Train)
        oof_train[val_idx, i] = model.predict_proba(X_va)[:, 1]

        # Test 예측 확률 저장
        test_preds_fold[:, fold] = model.predict_proba(X_test_final)[:, 1]

    # Test 예측 확률의 Fold 평균 (OOF Test)
    oof_test[:, i] = test_preds_fold.mean(axis=1)

print(" 메타 피처 생성 완료!")

# ---------------------------------------------------------
# 3. Meta-Learner(Logistic Regression) 학습 및 최적 Threshold 적용
# ---------------------------------------------------------
print("\n=== Meta-Learner(Stacking) 학습 및 평가 시작 ===")

# 메타 모델 정의 및 학습 (가중치 조정 없이 OOF 확률 자체의 선형 조합 학습)
meta_model = LogisticRegression(random_state=42, max_iter=1000)
meta_model.fit(oof_train, y_train)

# Meta-Learner 예측 확률
meta_pred_prob = meta_model.predict_proba(oof_test)[:, 1]

# PR 커브 및 F1 최적 Decision Threshold 탐색
precision_vec, recall_vec, thresholds = precision_recall_curve(y_test, meta_pred_prob)

denom = precision_vec + recall_vec
f1_scores = np.divide(
    2 * (precision_vec * recall_vec),
    denom,
    out=np.zeros_like(denom),
    where=denom != 0
)

best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx] if best_idx < len(thresholds) else 0.5

# 최적 Threshold 적용
meta_pred = (meta_pred_prob >= best_threshold).astype(int)

# ---------------------------------------------------------
# 4. 최종 스태킹 지표 산출 및 비교
# ---------------------------------------------------------
pr_auc_val = auc(recall_vec, precision_vec)
roc_auc_val = roc_auc_score(y_test, meta_pred_prob)
tn, fp, fn, tp = confusion_matrix(y_test, meta_pred).ravel()
total_errors = fn + fp
error_rate = (total_errors / len(y_test)) * 100

f1_val = f1_score(y_test, meta_pred, zero_division=0)
rec_val = recall_score(y_test, meta_pred, zero_division=0)
prec_val = precision_score(y_test, meta_pred, zero_division=0)

print(f"\n[Stacking Ensemble 최종 평가 결과 (Best Threshold: {best_threshold:.4f})]")
print(f"F1-Score            : {f1_val:.4f}")
print(f"PR-AUC Score        : {pr_auc_val:.4f}")
print(f"ROC-AUC Score       : {roc_auc_val:.4f}")
print(f"Precision           : {prec_val:.4f}")
print(f"Recall              : {rec_val:.4f}")
print(f"총 오류 건수(FN+FP)  : {total_errors:,}건 (미검출 FN: {fn}건, 과잉검출 FP: {fp}건)")
print(f"오탐 비율(%)        : {error_rate:.2f}%")

print('\n[Classification Report]')
print(classification_report(y_test, meta_pred, zero_division=0))

# 기존 Optuna 단일 모델 요약표와 Stacking 결과 비교용 DataFrame 추가
stacking_res = pd.DataFrame([{
    'Model': 'Stacking (Meta-Learner)',
    'Best Threshold': round(best_threshold, 4),
    'F1-Score': f1_val,
    'PR-AUC': pr_auc_val,
    'ROC-AUC': roc_auc_val,
    'Precision': prec_val,
    'Recall': rec_val,
    '오류건수(FN+FP)': total_errors,
    '오탐비율(%)': error_rate,
    '미검출(FN)': fn,
    '과잉검출(FP)': fp
}])

final_summary_df = pd.concat([tuned_summary_df, stacking_res], ignore_index=True).sort_values(by='F1-Score', ascending=False).reset_index(drop=True)

print('\n' + '='*75)
print('[단일 최적화 모델 vs Stacking 앙상블 최종 종합 비교 요약]')
print('='*75)
print(final_summary_df.to_string(index=False))

In [ ]:
final_summary_df